# Static accuracy: aggregate report

Accumulated static positioning accuracy over many runs. In the
default `GROUP = 'pose'` mode this is the thesis headline analysis:
all comparable runs at one target, averaged across runs with t-based
95 % confidence intervals. `GROUP = 'all'` pools every completed run
regardless of pose for a workspace-wide performance picture, always
accompanied by the per-pose breakdown so poses are never silently
mixed. `MATCH_CYCLES` filters to a single iteration count when wanted.

Protocol: 30 cycles per run (ISO 9283), 3 runs minimum, 5 preferred,
re-homed between runs.

*Note: the kernel imports `volcaniarm_calibration` through a `.pth` file; restart the kernel after changing the package code. Every figure is also saved to `notebooks/figures/` as a 300 dpi PNG and a vector PDF, ready for the thesis.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    load_runs, select_comparable_runs, filter_runs_by_goals,
    filter_runs_by_cycles, concat_runs, mount_key,
    apply_style, save_fig, run_short, per_axis_residuals_mm,
    PRIMARY, ACCENT, RUN_COLORS, FIG_FULL, FIG_TALL, FIG_SQUARE,
    summary, cross_run_summary, per_point_accuracy,
    threshold_zone, threshold_color,
)
apply_style()

TEST_NAME = 'static_accuracy'

GROUP = 'pose'           # 'pose' = same-pose averaging; 'all' = every pose pooled
POSE = None              # (y, z) used by GROUP='pose'; None = pose with most runs
MATCH_CYCLES = None      # e.g. 30 to keep only 30-cycle runs; None = include all
RUN_DIRS = None          # pin the exact run set for final thesis figures
ALLOW_MOUNT_KEYS = None  # merge legacy mount keys known to be identical
MIN_RUNS = 3             # protocol target; fewer runs still compute

all_runs = load_runs(TEST_NAME, run_dirs=RUN_DIRS)
HAVE = bool(all_runs)
if not HAVE:
    print('No completed runs on disk; record some from the calibration '
          'dashboard first.')

if HAVE:
    by_target = {}
    for r in all_runs:
        key = tuple(round(float(v), 3) for v in r['config']['goals'][0])
        by_target.setdefault(key, []).append(r)
    print('Targets on disk:')
    for key, rs in sorted(by_target.items()):
        print(f'  y={key[0]:+.3f} z={key[1]:.3f}: {len(rs)} run(s), '
              f'latest {rs[-1]["config"].get("run_id")}')
    print()

    if MATCH_CYCLES is not None:
        n_before = len(all_runs)
        all_runs = filter_runs_by_cycles(all_runs, MATCH_CYCLES)
        print(f'Cycle filter: kept {len(all_runs)}/{n_before} runs '
              f'with num_cycles == {MATCH_CYCLES}')
        if not all_runs:
            raise RuntimeError('no runs left after the cycle filter')

    if GROUP == 'pose':
        runs = (filter_runs_by_goals(all_runs, [POSE]) if POSE is not None
                else all_runs)
        if not runs:
            raise RuntimeError(f'no completed runs at pose {POSE}; '
                               'see the list above')
        runs = select_comparable_runs(runs, allow_mount_keys=ALLOW_MOUNT_KEYS)
    elif GROUP == 'all':
        runs = select_comparable_runs(all_runs,
                                      allow_mount_keys=ALLOW_MOUNT_KEYS,
                                      match_goals=False)
    else:
        raise ValueError(f"unknown GROUP {GROUP!r}")
    df = concat_runs(runs)
    n_targets = df[['goal_y', 'goal_z']].drop_duplicates().shape[0]
    goal_y, goal_z = runs[-1]['config']['goals'][0]
    run_ids = list(dict.fromkeys(df['run_id']))

    rows = []
    for run_id, g in df.groupby('run_id', sort=False):
        from volcaniarm_calibration.analysis import summary as _summary
        s = _summary(g['d_error'].dropna()).in_mm()
        cfg = next(r['config'] for r in runs
                   if r['config'].get('run_id') == run_id)
        py, pz = cfg['goals'][0]
        rows.append({'run': run_short(run_id),
                     'target': f'({py:+.2f}, {pz:.3f})',
                     'cycles': cfg.get('num_cycles'),
                     'mount': mount_key(cfg),
                     'n': s.n, 'mean [mm]': round(s.mean, 2),
                     'std [mm]': round(s.std, 2),
                     'ci95 [mm]': round(s.ci95, 2),
                     'worst [mm]': round(s.worst, 2)})
    per_run_table = pd.DataFrame(rows)

    if n_targets == 1:
        print(f'Target: y = {goal_y:.3f} m, z = {goal_z:.3f} m')
    else:
        print(f'Scope: all poses pooled ({n_targets} poses)')
    print(f'Runs aggregated: {len(runs)}'
          + ('' if len(runs) >= MIN_RUNS else
             f'  (below the protocol target of {MIN_RUNS})'))
    display(per_run_table)

## Headline statistics

The across-run mean with its t-based confidence interval is the
number to quote (degrees of freedom = runs - 1). The pooled standard
deviation is the positioning precision. Weeding zones: 10 mm
acceptable, 30 mm marginal.

In [ ]:
if HAVE:
    per_run_vals = [g['d_error'].dropna().tolist()
                    for _, g in df.groupby('run_id', sort=False)]
    cr = cross_run_summary(per_run_vals)

    if cr.n_runs >= 2:
        print(f'Across-run mean residual:  {cr.mean * 1000:+7.2f} mm '
              f'+/- {cr.ci95 * 1000:.2f} mm (95 % CI, {cr.n_runs} runs)')
        print(f'Between-run std:           '
              f'{cr.std_between * 1000:7.2f} mm')
    else:
        print('Single run only: across-run CI is undefined; quote the '
              'pooled statistics and state n_runs = 1.')
    pooled = cr.pooled.in_mm()
    print(f'Pooled precision (std):    {pooled.std:7.2f} mm  '
          f'({pooled.n} cycles pooled)')
    print(f'Pooled worst residual:     {pooled.worst:7.2f} mm')
    print()
    print(f'Weeding zone of |mean bias|: '
          f'{threshold_zone(abs(cr.mean) * 1000)}')
    print(f'Weeding zone of precision:   {threshold_zone(pooled.std)}')
    if n_targets > 1:
        print()
        print(f'NOTE: {n_targets} poses pooled (GROUP = all). The '
              'across-run numbers mix per-pose biases; use the per-pose '
              'breakdown below for pose-specific statements.')

## Per-pose breakdown

One row per commanded pose. A single row under `GROUP = 'pose'`;
under `GROUP = 'all'` this is the per-pose accuracy summary across
the whole dataset.

In [ ]:
if HAVE:
    breakdown = per_point_accuracy(df)
    display(breakdown)

## Per-run distribution

One box per run with the cycles overlaid; the dashed line is the
kinematic prediction. Box spread = within-run precision; box
displacement = session-to-session variation.

In [ ]:
if HAVE:
    data = [df[df['run_id'] == rid]['d_error'].dropna().to_numpy() * 1000.0
            for rid in run_ids]
    fig, ax = plt.subplots(figsize=FIG_TALL)
    ax.boxplot(data, positions=range(len(data)), widths=0.5,
               showfliers=False, medianprops={'color': '#555555'})
    rng = np.random.default_rng(0)
    for i, vals in enumerate(data):
        xj = i + rng.uniform(-0.12, 0.12, len(vals))
        ax.plot(xj, vals, 'o', ms=6, color=RUN_COLORS[i % len(RUN_COLORS)],
                mec='white', mew=0.8, alpha=0.9, zorder=5)
    ax.axhline(0.0, color='#888888', ls='--', lw=1.2,
               label='kinematic prediction')
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels([run_short(r) for r in run_ids], rotation=15)
    ax.set_xlabel('run (date time)')
    ax.set_ylabel('residual e  [mm]')
    ax.set_title('Static accuracy residual per run')
    ax.legend(loc='best')
    save_fig(fig, 'static_accuracy_aggregate/per_run_box')

## Residual distribution per run

Violin plot of each run's full residual distribution (white dot:
median; bar: interquartile range); differences in spread or skew
between sessions are visible directly.

In [ ]:
if HAVE:
    fig, ax = plt.subplots(figsize=FIG_TALL)
    parts = ax.violinplot(data, positions=range(len(data)),
                          showmedians=False, showextrema=False, widths=0.7)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(RUN_COLORS[i % len(RUN_COLORS)])
        body.set_alpha(0.6)
        body.set_edgecolor('#555555')
    for i, vals in enumerate(data):
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        ax.vlines(i, q1, q3, color='#333333', lw=4)
        ax.plot(i, med, 'o', ms=5, color='white', mec='#333333', zorder=5)
    ax.axhline(0.0, color='#888888', ls='--', lw=1.2,
               label='kinematic prediction')
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels([run_short(r) for r in run_ids], rotation=15)
    ax.set_xlabel('run (date time)')
    ax.set_ylabel('residual e  [mm]')
    ax.set_title('Residual distribution per run')
    ax.legend(loc='best')
    save_fig(fig, 'static_accuracy_aggregate/per_run_violin')

## Pooled residual distribution

All cycles from all runs with a normal curve fitted to the pooled
mean and standard deviation.

In [ ]:
if HAVE:
    from scipy.stats import norm

    e_all = df['d_error'].dropna().to_numpy() * 1000.0
    mu, sd = float(e_all.mean()), float(e_all.std(ddof=1))
    fig, ax = plt.subplots(figsize=FIG_FULL)
    counts, bins, _ = ax.hist(e_all, bins=15, color=PRIMARY,
                              edgecolor='white', alpha=0.85)
    if sd > 0:
        xs = np.linspace(bins[0], bins[-1], 200)
        # ax.plot(xs, norm.pdf(xs, mu, sd) * len(e_all) * (bins[1] - bins[0]),
        #         '-', color='#555555', lw=2, label='normal fit')
    ax.axvline(mu, color='#d04b4b', ls='--', lw=1.5,
               label=f'mean {mu:+.2f} mm')
    ax.set_xlabel('residual e  [mm]')
    ax.set_ylabel('cycles')
    ax.set_title('Distribution of the accuracy residual')
    ax.legend(loc='best')
    save_fig(fig, 'static_accuracy_aggregate/histogram')

## Precision vs application tolerance

Empirical CDF of the deviation about each run's own mean (bias
removed) against the weeding thresholds; reads as "X % of cycles
landed within Y mm of the run mean".

In [ ]:
if HAVE:
    dev = np.concatenate([
        (g['d_error'] - g['d_error'].mean()).dropna().to_numpy() * 1000.0
        for _, g in df.groupby('run_id', sort=False)])
    xs = np.sort(np.abs(dev))
    pct = np.arange(1, len(xs) + 1) / len(xs) * 100.0
    fig, ax = plt.subplots(figsize=FIG_FULL)
    ax.step(xs, pct, where='post', color=PRIMARY, lw=2.0)
    ax.axvline(10.0, color='#2e9c4a', ls='--', lw=1.4,
               label='acceptable (10 mm)')
    ax.axvline(30.0, color='#c79a3a', ls='--', lw=1.4,
               label='marginal (30 mm)')
    ax.set_xlim(0, max(12.0, float(xs.max()) * 1.15))
    ax.set_ylim(0, 102)
    ax.set_xlabel('|deviation about run mean|  [mm]')
    ax.set_ylabel('cycles within  [%]')
    ax.set_title('Precision against the weeding tolerance')
    ax.legend(loc='lower right')
    save_fig(fig, 'static_accuracy_aggregate/ecdf_tolerance')

## Convergence of the mean

Running mean per run as cycles accumulate; the empirical
justification for the 30-cycle protocol.

In [ ]:
if HAVE:
    fig, ax = plt.subplots(figsize=FIG_FULL)
    for i, rid in enumerate(run_ids):
        e = df[df['run_id'] == rid]['d_error'].dropna().to_numpy() * 1000.0
        xc = np.arange(1, len(e) + 1)
        ax.plot(xc, np.cumsum(e) / xc, '-o', ms=4, lw=1.4,
                color=RUN_COLORS[i % len(RUN_COLORS)], mec='white',
                mew=0.5, label=run_short(rid))
    ax.axhline(cr.mean * 1000.0, color='#555555', ls='--', lw=1.2,
               label='across-run mean')
    ax.set_xlabel('cycles included')
    ax.set_ylabel('running mean residual  [mm]')
    ax.set_title('Convergence of the mean residual')
    ax.legend(loc='best')
    save_fig(fig, 'static_accuracy_aggregate/convergence')

## Accumulated performance over sessions

Each run's mean residual with its within-run 95 % CI, in recording
order. Flat means stable calibration; a step means something changed
between sessions (camera, mounts, homing).

In [ ]:
if HAVE:
    means = [s.mean for s in cr.per_run]
    cis = [s.ci95 for s in cr.per_run]
    xs = np.arange(len(means))
    fig, ax = plt.subplots(figsize=FIG_FULL)
    ax.errorbar(xs, np.array(means) * 1000.0,
                yerr=np.array(cis) * 1000.0, fmt='o-', ms=7, lw=1.4,
                color=PRIMARY, mec='white', mew=0.8, capsize=4,
                ecolor='#555555')
    ax.axhline(cr.mean * 1000.0, color='#555555', ls='--', lw=1.2,
               label='across-run mean')
    ax.set_xticks(xs)
    ax.set_xticklabels([run_short(r) for r in run_ids], rotation=15)
    ax.set_xlabel('run (date time)')
    ax.set_ylabel('mean residual +/- 95 % CI  [mm]')
    ax.set_title('Mean residual across sessions')
    ax.legend(loc='best')
    save_fig(fig, 'static_accuracy_aggregate/sessions')

## Per-axis decomposition

Base-relative world Y and Z residual per cycle in chronological order;
vertical lines separate runs.

In [ ]:
if HAVE:
    df_sorted = df.reset_index(drop=True)
    ry, rz = per_axis_residuals_mm(df_sorted)
    xs = np.arange(1, len(df_sorted) + 1)
    fig, ax = plt.subplots(figsize=FIG_TALL)
    ax.plot(xs, ry, '-o', ms=4, lw=1.2, color=PRIMARY, mec='white',
            mew=0.5, label='Y residual')
    ax.plot(xs, rz, '-s', ms=4, lw=1.2, color=ACCENT, mec='white',
            mew=0.5, label='Z residual')
    ax.axhline(0.0, color='#888888', ls='--', lw=1.0)
    boundaries = df_sorted['run_id'].ne(df_sorted['run_id'].shift())
    for i in np.flatnonzero(boundaries.to_numpy())[1:]:
        ax.axvline(i + 0.5, color='#cccccc', lw=1.0)
    ax.set_xlabel('cycle (chronological, all runs)')
    ax.set_ylabel('residual  [mm]')
    ax.set_title('Per-axis positioning residual')
    ax.legend(loc='best')
    save_fig(fig, 'static_accuracy_aggregate/per_axis')

## Normality check

Quantile-quantile plot of the bias-removed deviations against a
normal distribution; points on the line support the t-based
confidence intervals.

In [ ]:
if HAVE:
    from scipy import stats as sps

    fig, ax = plt.subplots(figsize=FIG_SQUARE)
    sps.probplot(dev, dist='norm', plot=ax)
    ax.get_lines()[0].set(marker='o', markersize=6,
                          markerfacecolor=PRIMARY,
                          markeredgecolor='white', markeredgewidth=0.6,
                          linestyle='none')
    ax.get_lines()[1].set(color='#555555', linewidth=1.6)
    ax.set_title('Normal Q-Q plot of the deviations')
    ax.set_xlabel('theoretical quantiles')
    ax.set_ylabel('deviation about run mean  [mm]')
    save_fig(fig, 'static_accuracy_aggregate/qq')

## Ranked per-pose accuracy (GROUP = 'all')

Poses sorted by residual magnitude with confidence intervals,
coloured by weeding zone. Rendered only when more than one pose is in
scope.

In [ ]:
if HAVE and n_targets > 1:
    srt = breakdown.reindex(
        breakdown['mean_mm'].abs().sort_values().index)
    labels = [f"({r['goal_y']:+.2f}, {r['goal_z']:.3f})"
              for _, r in srt.iterrows()]
    colors = [threshold_color(abs(v)) for v in srt['mean_mm']]
    fig, ax = plt.subplots(figsize=(6.3, 0.5 * len(srt) + 1.6))
    ax.barh(range(len(srt)), srt['mean_mm'], xerr=srt['ci95_mm'],
            color=colors, edgecolor='white', height=0.6,
            error_kw={'ecolor': '#555555', 'capsize': 3})
    ax.axvline(0.0, color='#888888', lw=1.0)
    ax.set_yticks(range(len(srt)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('mean residual +/- 95 % CI  [mm]')
    ax.set_title('Poses ranked by residual magnitude')
    save_fig(fig, 'static_accuracy_aggregate/ranked_poses')

## Interpretation

Quote the across-run mean with its t-based confidence interval as the
accuracy figure and the pooled standard deviation as the precision.
While the URDF tag mounts carry placeholder values the mean reflects
the fiducial model, not the arm (see CALIBRATION.md, Mount bias);
after the mounts are fixed it is the absolute pose accuracy in the
ISO 9283 AP sense. The sessions figure shows whether the calibration
state was stable over the campaign.